# Лабораторная работа 3: Максимальный поток

**Формат:** DIMACS max-flow (QPBO) с отрицательными ёмкостями.  
Применяется преобразование Бороша–Хаммера → стандартный max-flow.  
**Алгоритм:** Эдмондса–Карпа (BFS Ford–Fulkerson), O(VE²).

In [1]:
import time
from collections import deque
from pathlib import Path

BASE_DIR = Path("/home/goringich/Desktop/hse/optimizations/lab3")
print("BASE_DIR:", BASE_DIR)


BASE_DIR: /home/goringich/Desktop/hse/optimizations/lab3


In [ ]:
# ── Разбор DIMACS ──────────────────────────────────────────────────────
def parse_dimacs(filepath):
    """Вернуть (n_nodes, source, sink, arcs)."""
    n_nodes, source, sink, arcs = 0, None, None, []
    with open(filepath) as f:
        for line in f:
            p = line.split()
            if not p or p[0] == "c":
                continue
            if p[0] == "p":
                n_nodes = int(p[2])
            elif p[0] == "n":
                if p[2] == "s": source = int(p[1])
                elif p[2] == "t": sink   = int(p[1])
            elif p[0] == "a":
                arcs.append((int(p[1]), int(p[2]), int(p[3])))
    return n_nodes, source, sink, arcs


# ── Преобразование Бороша–Хаммера ───────────────────────────────────────
def boros_hammer(arcs, source, sink):
    """Привести QPBO-граф к стандартному max-flow (неотрицательные ёмкости)."""
    src_cap, snk_cap, nbr = {}, {}, []
    for u, v, c in arcs:
        if u == source:   src_cap[v] = src_cap.get(v, 0) + c
        elif v == sink:   snk_cap[u] = snk_cap.get(u, 0) + c
        else:             nbr.append((u, v, c))
    constant, terminal = 0, []
    for v in set(src_cap) | set(snk_cap):
        ts, tt = src_cap.get(v, 0), snk_cap.get(v, 0)
        constant += min(ts, tt)
        if ts - tt > 0: terminal.append((source, v, ts - tt))
        if tt - ts > 0: terminal.append((v, sink,  tt - ts))
    return terminal + nbr, constant


# ── Граф остаточных пропускных способностей ────────────────────────────
def build_residual(arcs):
    cap = {}
    for u, v, c in arcs:
        cap.setdefault(u, {})
        cap.setdefault(v, {})
        cap[u][v] = cap[u].get(v, 0) + c
        if u not in cap[v]: cap[v][u] = 0
    return cap


# ── Алгоритм Эдмондса–Карпа ────────────────────────────────────────────
def _bfs(cap, s, t):
    parent = {s: -1}
    q = deque([s])
    while q:
        u = q.popleft()
        for v, r in cap.get(u, {}).items():
            if r > 0 and v not in parent:
                parent[v] = u
                if v == t: return parent
                q.append(v)
    return None

def edmonds_karp(cap, s, t):
    """Найти максимальный поток (изменяет cap на месте)."""
    mf = 0
    while True:
        parent = _bfs(cap, s, t)
        if parent is None: break
        pf, v = float('inf'), t
        while v != s:
            u = parent[v]; pf = min(pf, cap[u][v]); v = u
        v = t
        while v != s:
            u = parent[v]
            cap[u][v] -= pf
            cap[v][u] = cap[v].get(u, 0) + pf
            v = u
        mf += pf
    return mf


# ── Извлечение потоков по дугам ─────────────────────────────────────────
def extract_arc_flows(reparam_arcs, residual_cap):
    """Вернуть {(u,v): flow} для дуг с ненулевым потоком."""
    orig = {}
    for u, v, c in reparam_arcs:
        orig[(u, v)] = orig.get((u, v), 0) + c
    return {(u, v): c - residual_cap.get(u, {}).get(v, 0)
            for (u, v), c in orig.items()
            if c - residual_cap.get(u, {}).get(v, 0) > 0}


# ── Решение одного файла ────────────────────────────────────────────────
def solve(filepath):
    t0 = time.perf_counter()
    n_nodes, src, snk, arcs = parse_dimacs(filepath)
    rep_arcs, const = boros_hammer(arcs, src, snk)
    cap = build_residual(rep_arcs)
    mf  = edmonds_karp(cap, src, snk)
    flows = extract_arc_flows(rep_arcs, cap)
    return {
        "max_flow":    mf,
        "constant":    const,
        "qpbo_energy": mf + const,
        "arc_flows":   flows,
        "source":      src,
        "sink":        snk,
        "elapsed":     time.perf_counter() - t0,
    }


print("Все функции определены.")


Все функции определены.


In [3]:
def collect_problems(base_dir):
    """Собрать все .bq.max файлы из уже распакованных директорий."""
    problems = []
    for ds_dir in sorted(base_dir.iterdir()):
        if not ds_dir.is_dir(): continue
        for grp_dir in sorted(ds_dir.iterdir()):
            if not grp_dir.is_dir(): continue
            for pf in sorted(grp_dir.glob("*.bq.max")):
                problems.append((ds_dir.name, grp_dir.name, pf))
    return problems

all_problems = collect_problems(BASE_DIR)

from collections import Counter
cnt = Counter(d for d, g, p in all_problems)
print(f"Всего задач: {len(all_problems)}")
for ds, n in sorted(cnt.items()):
    print(f"  {ds}: {n} задач")


Всего задач: 16200
  dimacs_car: 9000 задач
  dimacs_matching: 1200 задач
  dimacs_motor: 6000 задач


In [4]:
TIMEOUT_SEC        = 30.0  # лимит на одну задачу
MAX_DETAIL_PER_GRP = 3     # подробный вывод для первых N задач в каждой группе

def run_all(problems):
    grouped = {}
    for ds, grp, fpath in problems:
        grouped.setdefault((ds, grp), []).append(fpath)

    solved = skipped = 0

    for (ds, grp), fpaths in sorted(grouped.items()):
        rows   = []
        detail = 0

        for fpath in sorted(fpaths):
            try:
                res = solve(fpath)
                if res['elapsed'] > TIMEOUT_SEC:
                    skipped += 1; continue
                solved += 1
                rows.append({
                    'name':        fpath.name,
                    'max_flow':    res['max_flow'],
                    'qpbo_energy': res['qpbo_energy'],
                    'n_arcs':      len(res['arc_flows']),
                    'elapsed':     res['elapsed'],
                })
                # ── подробный вывод ──────────────────────────────────
                if detail < MAX_DETAIL_PER_GRP:
                    detail += 1
                    src, snk = res['source'], res['sink']
                    print(f"\n  [{ds}/{grp}] {fpath.name}")
                    print(f"    Максимальный поток : {res['max_flow']}")
                    if res['constant']:
                        print(f"    Константа BH       : {res['constant']}")
                        print(f"    Опт. QPBO-энергия  : {res['qpbo_energy']}")
                    if res['arc_flows']:
                        print(f"    Дуги с ненулевым потоком:")
                        for (u, v), fl in sorted(res['arc_flows'].items()):
                            tag = ' [исток→]' if u == src else (' [→сток]' if v == snk else '')
                            print(f"      {u:>4} → {v:<4}  поток = {fl}{tag}")
                    else:
                        print("    Ненулевых дуг нет.")
            except Exception as e:
                skipped += 1
                print(f"  ОШИБКА {fpath.name}: {e}")

        # ── сводная таблица группы ────────────────────────────────────
        if rows:
            SEP = '─' * 84
            print(f"\n{SEP}")
            print(f"  {ds}/{grp}  ({len(rows)} задач)")
            print(f"{SEP}")
            print(f"  {'Файл':<42} {'Поток':>10} {'QPBO E':>10} {'Дуг≠0':>6} {'мс':>7}")
            print(SEP)
            for r in rows:
                print(f"  {r['name']:<42} {r['max_flow']:>10} {r['qpbo_energy']:>10} "
                      f"{r['n_arcs']:>6} {r['elapsed']*1000:>7.1f}")

    print(f"\n{'='*84}")
    print(f"  ИТОГО: решено {solved}, пропущено {skipped}")
    print(f"{'='*84}")


t0 = time.perf_counter()
run_all(all_problems)
print(f"\nОбщее время: {time.perf_counter() - t0:.1f} с")



  [dimacs_car/car1.dd] qpbo_problem_1.cleanrescale.bq.max
    Максимальный поток : 14372
    Константа BH       : -29454
    Опт. QPBO-энергия  : -15082
    Дуги с ненулевым потоком:
         1 → 3     поток = 913 [исток→]
         1 → 4     поток = 378 [исток→]
         1 → 6     поток = 1748 [исток→]
         1 → 8     поток = 557 [исток→]
         1 → 9     поток = 1231 [исток→]
         1 → 11    поток = 294 [исток→]
         1 → 12    поток = 1460 [исток→]
         1 → 13    поток = 197 [исток→]
         1 → 19    поток = 1571 [исток→]
         1 → 21    поток = 210 [исток→]
         1 → 24    поток = 1348 [исток→]
         1 → 26    поток = 481 [исток→]
         1 → 29    поток = 2468 [исток→]
         1 → 33    поток = 194 [исток→]
         1 → 35    поток = 260 [исток→]
         1 → 36    поток = 1062 [исток→]
         3 → 5     поток = 725
         3 → 10    поток = 3028
         3 → 27    поток = 162
         4 → 26    поток = 771
         5 → 2     поток = 1348 [→сток]
    


────────────────────────────────────────────────────────────────────────────────────
  dimacs_car/car12.dd  (300 задач)
────────────────────────────────────────────────────────────────────────────────────
  Файл                                            Поток     QPBO E  Дуг≠0      мс
────────────────────────────────────────────────────────────────────────────────────
  qpbo_problem_1.cleanrescale.bq.max              12802     -30908     71     0.6
  qpbo_problem_10.cleanrescale.bq.max              8584     -25504     43     0.4
  qpbo_problem_100.cleanrescale.bq.max                0     -27776      0     0.1
  qpbo_problem_101.cleanrescale.bq.max                0     -27776      0     0.1
  qpbo_problem_102.cleanrescale.bq.max                0     -27776      0     0.1
  qpbo_problem_103.cleanrescale.bq.max                0     -27776      0     0.1
  qpbo_problem_104.cleanrescale.bq.max                0     -27776      0     0.1
  qpbo_problem_105.cleanrescale.bq.max               


────────────────────────────────────────────────────────────────────────────────────
  dimacs_car/car13.dd  (300 задач)
────────────────────────────────────────────────────────────────────────────────────
  Файл                                            Поток     QPBO E  Дуг≠0      мс
────────────────────────────────────────────────────────────────────────────────────
  qpbo_problem_1.cleanrescale.bq.max              13380     -41440     99     0.9
  qpbo_problem_10.cleanrescale.bq.max                 0     -32216      0     0.2
  qpbo_problem_100.cleanrescale.bq.max                0     -33216      0     0.2
  qpbo_problem_101.cleanrescale.bq.max                0     -36378      0     0.2
  qpbo_problem_102.cleanrescale.bq.max                0     -33216      0     0.2
  qpbo_problem_103.cleanrescale.bq.max                0     -33216      0     0.2
  qpbo_problem_104.cleanrescale.bq.max                0     -33216      0     0.2
  qpbo_problem_105.cleanrescale.bq.max               


────────────────────────────────────────────────────────────────────────────────────
  dimacs_car/car15.dd  (300 задач)
────────────────────────────────────────────────────────────────────────────────────
  Файл                                            Поток     QPBO E  Дуг≠0      мс
────────────────────────────────────────────────────────────────────────────────────
  qpbo_problem_1.cleanrescale.bq.max              16226     -29784     90     0.7
  qpbo_problem_10.cleanrescale.bq.max              1446     -37224     10     0.2
  qpbo_problem_100.cleanrescale.bq.max              380     -39610      6     0.3
  qpbo_problem_101.cleanrescale.bq.max             1868     -32098     10     0.3
  qpbo_problem_102.cleanrescale.bq.max                0     -36020      0     0.2
  qpbo_problem_103.cleanrescale.bq.max                0     -36668      0     0.2
  qpbo_problem_104.cleanrescale.bq.max                0     -36020      0     0.2
  qpbo_problem_105.cleanrescale.bq.max               


  [dimacs_car/car17.dd] qpbo_problem_1.cleanrescale.bq.max
    Максимальный поток : 12378
    Константа BH       : -39850
    Опт. QPBO-энергия  : -27472
    Дуги с ненулевым потоком:
         1 → 3     поток = 2680 [исток→]
         1 → 5     поток = 344 [исток→]
         1 → 8     поток = 583 [исток→]
         1 → 9     поток = 443 [исток→]
         1 → 11    поток = 468 [исток→]
         1 → 12    поток = 51 [исток→]
         1 → 15    поток = 650 [исток→]
         1 → 18    поток = 1012 [исток→]
         1 → 22    поток = 584 [исток→]
         1 → 26    поток = 460 [исток→]
         1 → 27    поток = 337 [исток→]
         1 → 31    поток = 115 [исток→]
         1 → 34    поток = 337 [исток→]
         1 → 41    поток = 256 [исток→]
         1 → 43    поток = 462 [исток→]
         1 → 44    поток = 73 [исток→]
         1 → 50    поток = 1203 [исток→]
         1 → 51    поток = 2000 [исток→]
         1 → 52    поток = 320 [исток→]
         3 → 23    поток = 2680
         4 → 2     по

...


────────────────────────────────────────────────────────────────────────────────────
  dimacs_motor/motor11.dd  (300 задач)
────────────────────────────────────────────────────────────────────────────────────
  Файл                                            Поток     QPBO E  Дуг≠0      мс
────────────────────────────────────────────────────────────────────────────────────
  qpbo_problem_1.cleanrescale.bq.max              11734     -17840     67     0.5
  qpbo_problem_10.cleanrescale.bq.max              5646     -20006     28     0.2
  qpbo_problem_100.cleanrescale.bq.max                0     -28894      0     0.1
  qpbo_problem_101.cleanrescale.bq.max                0     -28894      0     0.1
  qpbo_problem_102.cleanrescale.bq.max                0     -28894      0     0.1
  qpbo_problem_103.cleanrescale.bq.max                0     -28894      0     0.1
  qpbo_problem_104.cleanrescale.bq.max                0     -28894      0     0.1
  qpbo_problem_105.cleanrescale.bq.max           


────────────────────────────────────────────────────────────────────────────────────
  dimacs_motor/motor14.dd  (300 задач)
────────────────────────────────────────────────────────────────────────────────────
  Файл                                            Поток     QPBO E  Дуг≠0      мс
────────────────────────────────────────────────────────────────────────────────────
  qpbo_problem_1.cleanrescale.bq.max              12470     -31292     88     0.7
  qpbo_problem_10.cleanrescale.bq.max             19398     -11988     99     0.9
  qpbo_problem_100.cleanrescale.bq.max             2620     -25376     21     0.2
  qpbo_problem_101.cleanrescale.bq.max             7492     -27012     60     0.6
  qpbo_problem_102.cleanrescale.bq.max                0     -30176      0     0.2
  qpbo_problem_103.cleanrescale.bq.max             6770     -24686     18     0.2
  qpbo_problem_104.cleanrescale.bq.max            19080     -10372     79     0.6
  qpbo_problem_105.cleanrescale.bq.max           


────────────────────────────────────────────────────────────────────────────────────
  dimacs_motor/motor17.dd  (300 задач)
────────────────────────────────────────────────────────────────────────────────────
  Файл                                            Поток     QPBO E  Дуг≠0      мс
────────────────────────────────────────────────────────────────────────────────────
  qpbo_problem_1.cleanrescale.bq.max              19802     -45530    100     1.1
  qpbo_problem_10.cleanrescale.bq.max              8150     -50086     24     0.3
  qpbo_problem_100.cleanrescale.bq.max             1734     -53458      8     0.2
  qpbo_problem_101.cleanrescale.bq.max                0     -52994      0     0.2
  qpbo_problem_102.cleanrescale.bq.max             1734     -51168      6     0.2
  qpbo_problem_103.cleanrescale.bq.max                0     -52898      0     0.2
  qpbo_problem_104.cleanrescale.bq.max             3724     -48760      6     0.2
  qpbo_problem_105.cleanrescale.bq.max           


────────────────────────────────────────────────────────────────────────────────────
  dimacs_motor/motor20.dd  (300 задач)
────────────────────────────────────────────────────────────────────────────────────
  Файл                                            Поток     QPBO E  Дуг≠0      мс
────────────────────────────────────────────────────────────────────────────────────
  qpbo_problem_1.cleanrescale.bq.max              12462     -50296     70     0.7
  qpbo_problem_10.cleanrescale.bq.max              8492     -42132     26     0.3
  qpbo_problem_100.cleanrescale.bq.max                0     -45894      0     0.2
  qpbo_problem_101.cleanrescale.bq.max                0     -45894      0     0.2
  qpbo_problem_102.cleanrescale.bq.max                0     -45894      0     0.2
  qpbo_problem_103.cleanrescale.bq.max                0     -45894      0     0.2
  qpbo_problem_104.cleanrescale.bq.max                0     -45894      0     0.2
  qpbo_problem_105.cleanrescale.bq.max           


────────────────────────────────────────────────────────────────────────────────────
  dimacs_motor/motor8.dd  (300 задач)
────────────────────────────────────────────────────────────────────────────────────
  Файл                                            Поток     QPBO E  Дуг≠0      мс
────────────────────────────────────────────────────────────────────────────────────
  qpbo_problem_1.cleanrescale.bq.max              25296     -42254    111     1.1
  qpbo_problem_10.cleanrescale.bq.max              4860     -50242      6     0.2
  qpbo_problem_100.cleanrescale.bq.max                0     -50764      0     0.2
  qpbo_problem_101.cleanrescale.bq.max             4860     -45986      6     0.2
  qpbo_problem_102.cleanrescale.bq.max             5570     -48664     10     0.2
  qpbo_problem_103.cleanrescale.bq.max             4860     -45986      6     0.2
  qpbo_problem_104.cleanrescale.bq.max             5570     -48664     10     0.2
  qpbo_problem_105.cleanrescale.bq.max            